# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# (Colab) Lấy repo về và chuyển vào thư mục repo. Chạy local trong repo thì cell này không làm gì.
import os, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # giảm phân mảnh VRAM (đặt trước khi dùng CUDA)
REPO_URL = "https://github.com/2003congthinh/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if "google.colab" in sys.modules and not os.path.exists("scripts/score.py"):
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# peft 0.21.2 dừng (ImportError) khi thấy torchao < 0.16 lúc gắn LoRA vào Linear fp16 (LoRA ViT ở E3).
# Colab cài sẵn torchao 0.10.0; lab không dùng torchao → gỡ đi.
!pip -q uninstall -y torchao

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# Google Drive: sao lưu adapter, dự đoán và bằng chứng (session Colab miễn phí có thể bị ngắt giữa chừng).
# Session mới: restore() chép lại work/ và report/ từ Drive về repo để chạy tiếp, không phải huấn luyện lại.
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_DIR = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/day10_vlm")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)


def backup(path):
    """Chép file/thư mục `path` (đường dẫn tương đối trong repo) sang Drive."""
    if DRIVE_DIR is None:
        return
    path = Path(path)
    dst = DRIVE_DIR / path
    if path.is_dir():
        shutil.copytree(path, dst, dirs_exist_ok=True)
    elif path.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dst)


def restore():
    """Chép work/ và report/ từ Drive về repo (không ghi đè file đang có)."""
    if DRIVE_DIR is None:
        return
    n = 0
    for sub in ("work", "report"):
        src = DRIVE_DIR / sub
        for f in (src.rglob("*") if src.exists() else []):
            dst = Path(sub) / f.relative_to(src)
            if f.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
                n += 1
    print(f"restore: chép {n} file từ {DRIVE_DIR}")


restore()

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh

import json, math, random, re, subprocess, time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

from scripts.labcommon import CLASSES, read_jsonl, write_jsonl

DATA = Path("data")
SPLITS_DIR = DATA / "splits"
WORK = Path("work")
EVID = Path("report/evidence")
FIG = Path("report/figures")
for d in (WORK, EVID, FIG):
    d.mkdir(parents=True, exist_ok=True)
COLORS = {"pothole": "#e6194b", "crack": "#f58231", "manhole": "#4363d8"}

splits = {s: read_jsonl(SPLITS_DIR / f"{s}.jsonl") for s in ("train", "val", "test")}
META = json.loads((SPLITS_DIR / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", META["dataset_version"])
for s, rows in splits.items():
    c = Counter(o["label"] for r in rows for o in r["objects"])
    print(f"  {s:5s} {len(rows):5d} ảnh  {sum(c.values()):5d} box  "
          + "  ".join(f"{k}={c[k]}" for k in CLASSES))

# Không session nào được nằm ở hai split
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not sessions[a] & sessions[b], f"session trùng giữa {a} và {b}: {sessions[a] & sessions[b]}"
print("số session:", {s: len(v) for s, v in sessions.items()}, "→ không session nào nằm ở hai split")

# Nhãn lỗi: box có w = 0 hoặc h = 0 (prepare_data.py chỉ kiểm tra giá trị nằm trong [0,1])
bad = [(s, r["image_id"], o) for s, rows in splits.items() for r in rows for o in r["objects"]
       if o["yolo"][2] <= 0 or o["yolo"][3] <= 0]
print("box suy biến (w=0 hoặc h=0):", bad)


def yolo_to_pixels(yolo, width, height):
    """YOLO (xc, yc, w, h) chuẩn hoá [0,1] → (x1, y1, x2, y2) pixel (số thực)."""
    xc, yc, w, h = yolo
    return ((xc - w / 2) * width, (yc - h / 2) * height, (xc + w / 2) * width, (yc + h / 2) * height)


def load_image(entry):
    return Image.open(DATA / entry["image_path"]).convert("RGB")


def draw_boxes(ax, boxes, color=None, ls="-", lw=1.8, show_label=True):
    """Vẽ list[(label, (x1, y1, x2, y2) pixel)] lên một trục matplotlib đã imshow ảnh."""
    for lab, (x1, y1, x2, y2) in boxes:
        c = color or COLORS.get(lab, "lime")
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=c, lw=lw, ls=ls))
        if show_label:
            ax.text(x1, max(y1 - 3, 8), lab, color="white", fontsize=6,
                    bbox=dict(fc=c, ec="none", pad=0.8))


# 6 ảnh train, mỗi ảnh lấy từ một session khác nhau
rng = random.Random(0)
by_session = defaultdict(list)
for r in splits["train"]:
    by_session[r["session"]].append(r)
samples = [rng.choice(by_session[s]) for s in rng.sample(sorted(by_session), 6)]
fig, axes = plt.subplots(2, 3, figsize=(15, 6.2))
for ax, e in zip(axes.flat, samples):
    ax.imshow(load_image(e))
    ax.axis("off")
    draw_boxes(ax, [(o["label"], yolo_to_pixels(o["yolo"], e["width"], e["height"])) for o in e["objects"]])
    ax.set_title(f"{e['image_id']}  [{e['session']}]  {len(e['objects'])} box", fontsize=8)
fig.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in COLORS.items()], loc="lower center", ncol=3)
fig.suptitle("6 ảnh train (6 phiên quay khác nhau) với box gốc từ nhãn YOLO")
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIG / "data_samples.png", dpi=110)
plt.show()

# Phân bố lớp theo split và kích thước box
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.2))
x, wbar = np.arange(len(CLASSES)), 0.27
for i, (s, rows) in enumerate(splits.items()):
    c = Counter(o["label"] for r in rows for o in r["objects"])
    bars = a1.bar(x + (i - 1) * wbar, [c[k] for k in CLASSES], wbar, label=f"{s} ({len(rows)} ảnh)")
    a1.bar_label(bars, fontsize=7)
a1.set_xticks(x, CLASSES)
a1.set_ylabel("số box")
a1.set_title("Số box từng lớp theo split (chia theo phiên quay)")
a1.legend()
for k in CLASSES:
    areas = np.array([o["yolo"][2] * o["yolo"][3] * 100 for rows in splits.values() for r in rows
                      for o in r["objects"] if o["label"] == k])
    a2.hist(np.log10(np.clip(areas, 1e-3, None)), bins=40, alpha=0.55, color=COLORS[k],
            label=f"{k}: trung vị {np.median(areas):.2f}% khung hình")
a2.set_xlabel("log10(diện tích box, % khung hình)")
a2.set_ylabel("số box")
a2.set_title("Box rất nhỏ: phần lớn < 1% khung hình")
a2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "data_stats.png", dpi=110)
plt.show()

In [ ]:
# Câu 6: đo rò rỉ nếu chia NGẪU NHIÊN. Mỗi ảnh test tìm ảnh train gần nhất (thumbnail xám 32×18, khoảng cách L2).
# So sánh split theo phiên (của đề) với một split ngẫu nhiên cùng tỉ lệ ảnh test.
def thumbs(entries):
    return np.stack([np.asarray(load_image(e).convert("L").resize((32, 18)), dtype=np.float32).ravel() / 255
                     for e in entries])


def nn_dist(test_x, train_x):
    d = np.sqrt(np.clip((test_x ** 2).sum(1)[:, None] + (train_x ** 2).sum(1)[None, :] - 2 * test_x @ train_x.T, 0, None))
    return d.min(1), d.argmin(1)


def leak_check(seed=0):
    every = splits["train"] + splits["val"] + splits["test"]
    X = thumbs(every)
    n_tr, n_te = len(splits["train"]), len(splits["test"])
    idx_tr, idx_te = np.arange(n_tr), np.arange(len(every) - n_te, len(every))
    d_sess, nn_sess = nn_dist(X[idx_te], X[idx_tr])
    perm = np.random.default_rng(seed).permutation(len(every))
    r_te, r_tr = perm[:n_te], perm[n_te:n_te + n_tr]
    d_rand, nn_rand = nn_dist(X[r_te], X[r_tr])
    thr = np.percentile(d_sess, 5)        # "gần trùng" = gần hơn 95% cặp test–train của split theo phiên
    print(f"khoảng cách tới ảnh train gần nhất (trung vị): theo phiên {np.median(d_sess):.2f} | "
          f"ngẫu nhiên {np.median(d_rand):.2f}")
    print(f"tỉ lệ ảnh test có 'bản gần trùng' trong train (d < {thr:.2f}): theo phiên {np.mean(d_sess < thr):.1%} | "
          f"ngẫu nhiên {np.mean(d_rand < thr):.1%}")
    i = int(np.argmin(d_rand))
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))
    j = int(np.argmin(np.abs(d_sess - np.median(d_sess))))
    pairs = [(every[r_te[i]], f"test (chia ngẫu nhiên)"), (every[r_tr[nn_rand[i]]], f"train gần nhất: d={d_rand[i]:.2f}"),
             (every[idx_te[j]], "test (chia theo phiên)"), (every[idx_tr[nn_sess[j]]], f"train gần nhất: d={d_sess[j]:.2f}")]
    for ax, (e, t) in zip(axes, pairs):
        ax.imshow(load_image(e))
        ax.axis("off")
        ax.set_title(f"{t}\n{e['image_id']}", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / "leak_check.png", dpi=110)
    plt.show()
    return {"median_session": float(np.median(d_sess)), "median_random": float(np.median(d_rand)),
            "thr": float(thr), "near_dup_session": float(np.mean(d_sess < thr)),
            "near_dup_random": float(np.mean(d_rand < thr))}


LEAK = leak_check()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

IMG_W, IMG_H = 640, 360


def coord_scale(coord, width=IMG_W, height=IMG_H):
    """Hệ số (sx, sy) để đổi toạ độ chuẩn hoá [0,1] sang thang đầu ra ("norm1000" hoặc "pixel")."""
    return (1000, 1000) if coord == "norm1000" else (width, height)


def yolo_to_box(label, yolo, coord="norm1000", width=IMG_W, height=IMG_H):
    xc, yc, w, h = yolo
    sx, sy = coord_scale(coord, width, height)
    x1, y1 = round((xc - w / 2) * sx), round((yc - h / 2) * sy)
    x2, y2 = round((xc + w / 2) * sx), round((yc + h / 2) * sy)
    x1, x2 = min(max(x1, 0), sx), min(max(x2, 0), sx)
    y1, y2 = min(max(y1, 0), sy), min(max(y2, 0), sy)
    # Box rất mảnh có thể bị làm tròn thành x2 <= x1: nới 1 đơn vị, vì score.parse_output
    # coi CẢ output là hỏng nếu có một box suy biến — model không được học điều đó.
    if x2 <= x1:
        x1, x2 = (x1, x1 + 1) if x1 < sx else (x1 - 1, x1)
    if y2 <= y1:
        y1, y2 = (y1, y1 + 1) if y1 < sy else (y1 - 1, y1)
    return {"label": label, "bbox_2d": [int(x1), int(y1), int(x2), int(y2)]}


def objects_to_answer(objects, coord="norm1000", width=IMG_W, height=IMG_H):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord, width, height)
             for o in objects if o["yolo"][2] > 0 and o["yolo"][3] > 0]   # bỏ nhãn lỗi w=0/h=0
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))       # trên→dưới, trái→phải
    return json.dumps(boxes, separators=(",", ":"))


def box_to_pixels(bbox, coord="norm1000", width=IMG_W, height=IMG_H):
    """Ngược lại của yolo_to_box: bbox_2d theo thang `coord` → (x1, y1, x2, y2) pixel."""
    sx, sy = coord_scale(coord, width, height)
    x1, y1, x2, y2 = bbox
    return (x1 * width / sx, y1 * height / sy, x2 * width / sx, y2 * height / sy)


def make_prompt(coord="norm1000", width=IMG_W, height=IMG_H):
    if coord == "norm1000":
        scale = ("integers on a 0-1000 scale relative to the image size "
                 "(x = 1000 * pixel_x / image_width, y = 1000 * pixel_y / image_height); "
                 "(0,0) is the top-left corner and (1000,1000) is the bottom-right corner")
    else:
        scale = (f"absolute integer pixel coordinates of the {width}x{height} image "
                 f"(x from 0 to {width}, y from 0 to {height}); (0,0) is the top-left corner")
    return ("Locate every road damage in this dashcam image. There are exactly 3 classes: "
            "pothole, crack, manhole.\n"
            "Answer with ONLY a JSON list and nothing else (no markdown, no explanation), e.g. "
            '[{"label": "crack", "bbox_2d": [x1, y1, x2, y2]}]\n'
            f"bbox_2d is the box [x1, y1, x2, y2] in {scale}. "
            "Return [] if there is no damage.")


PROMPT = make_prompt("norm1000")
PROMPT_PIXEL = make_prompt("pixel")
print(PROMPT, "\n")

# Kiểm tra: YOLO → JSON 0–1000 → pixel phải trùng box gốc (đã kẹp vào khung ảnh); sai số chỉ do làm tròn số nguyên
def clamp_px(b):
    return (min(max(b[0], 0), IMG_W), min(max(b[1], 0), IMG_H), min(max(b[2], 0), IMG_W), min(max(b[3], 0), IMG_H))


all_objs = [o for rows in splits.values() for r in rows for o in r["objects"] if o["yolo"][2] > 0 and o["yolo"][3] > 0]
n_out = sum(clamp_px(yolo_to_pixels(o["yolo"], IMG_W, IMG_H)) != yolo_to_pixels(o["yolo"], IMG_W, IMG_H) for o in all_objs)
print(f"{n_out}/{len(all_objs)} box YOLO tràn ra ngoài khung ảnh → bị kẹp về [0, 1000]")
for coord in ("norm1000", "pixel"):
    err = max(max(abs(u - v) for u, v in zip(box_to_pixels(yolo_to_box(o["label"], o["yolo"], coord)["bbox_2d"], coord),
                                             clamp_px(yolo_to_pixels(o["yolo"], IMG_W, IMG_H))))
              for o in all_objs)
    print(f"[{coord}] sai số lớn nhất khi quy ngược về pixel ({len(all_objs)} box): {err:.3f} px")
    assert err < 1.0

ex = max(splits["train"], key=lambda r: len(r["objects"]))      # ảnh nhiều box nhất
ans = objects_to_answer(ex["objects"])
print("\nảnh", ex["image_id"], "→", ans)
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.imshow(load_image(ex))
ax.axis("off")
draw_boxes(ax, [(o["label"], yolo_to_pixels(o["yolo"], IMG_W, IMG_H)) for o in ex["objects"]],
           color="lime", lw=3.5, show_label=False)
draw_boxes(ax, [(b["label"], box_to_pixels(b["bbox_2d"])) for b in json.loads(ans)], ls="--", lw=1.5)
ax.set_title("Kiểm tra round-trip: xanh lá = box YOLO gốc, nét đứt = JSON 0–1000 quy ngược về pixel", fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "label_roundtrip.png", dpi=110)
plt.show()

In [ ]:
# Câu 3 (thang toạ độ): "oracle" = đáp án ĐÚNG HOÀN HẢO, chấm với --coord khớp và KHÔNG khớp.
# Không cần GPU: cho thấy riêng việc lệch thang đã làm F1 sụp thế nào.
def coord_mismatch_demo(split="test"):
    rows = splits[split]
    out = {}
    for label_coord in ("norm1000", "pixel"):
        path = WORK / f"oracle_{label_coord}.jsonl"
        write_jsonl(path, [{"image_id": r["image_id"], "raw": objects_to_answer(r["objects"], label_coord)}
                           for r in rows])
        for score_coord in ("norm1000", "pixel"):
            res = subprocess.run([sys.executable, "scripts/score.py", "--pred", str(path), "--split", split,
                                  "--coord", score_coord, "--name", f"oracle-{label_coord}-as-{score_coord}",
                                  "--out", str(WORK / f"oracle_{label_coord}_as_{score_coord}.json"),
                                  "--n-boot", "200"], capture_output=True, text=True, check=True)
            d = json.loads((WORK / f"oracle_{label_coord}_as_{score_coord}.json").read_text(encoding="utf-8"))
            out[(label_coord, score_coord)] = d
            print(f"nhãn {label_coord:8s} chấm --coord {score_coord:8s}: F1={d['micro']['f1']:.3f} "
                  f"P={d['micro']['precision']:.3f} R={d['micro']['recall']:.3f} meanIoU={d['mean_iou']:.3f}")
    return out


ORACLE = coord_mismatch_demo()

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

import gc

import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

from scripts.score import evaluate, iou_xyxy, parse_output

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_processor(model_id=MODEL_ID):
    processor = AutoProcessor.from_pretrained(model_id)
    processor.tokenizer.padding_side = "left"      # sinh theo batch: đệm trái để token cuối là token thật
    return processor


def load_base_model(model_id=MODEL_ID):
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        # transformers 5.19 so khớp tên module bằng re.match (neo ở ĐẦU tên) hoặc endswith, nên "visual" KHÔNG khớp
        # "model.visual.merger.linear_fc1" → ViT + projector bị nén 4-bit. Phải ghi đủ tiền tố "model.visual".
        llm_int8_skip_modules=["model.visual", "lm_head"])   # ViT (kể cả projector) và lm_head giữ fp16
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quant, dtype=torch.float16, device_map={"": 0})
    q4 = [n for n, m in model.named_modules() if type(m).__name__ == "Linear4bit"]
    assert not [n for n in q4 if n.startswith("model.visual")], "ViT/projector bị lượng tử hoá 4-bit: kiểm tra llm_int8_skip_modules"
    print(f"{len(q4)} Linear 4-bit (phải là 252, chỉ trong LLM); model.visual giữ fp16")
    return model


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def user_message(image, prompt):
    return {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}


@torch.no_grad()
def generate_batch(model, processor, images, prompt, max_new_tokens=256):
    messages = [[user_message(im, prompt)] for im in images]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt", padding=True).to(model.device)
    with torch.autocast(model.device.type, dtype=torch.float16, enabled=model.device.type == "cuda"):
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def read_done(path):
    """Đọc các dòng đã ghi (bỏ dòng cuối bị cắt dở nếu session chết giữa lúc ghi)."""
    rows = []
    if Path(path).exists():
        for line in Path(path).read_text(encoding="utf-8").splitlines():
            try:
                rows.append(json.loads(line))
            except ValueError:
                pass
    return rows


def predict_split(model, processor, entries, out_path, prompt=None, batch_size=4, max_new_tokens=256, log_every=10):
    """Suy luận (greedy) cho `entries`, ghi {"image_id", "raw"} mỗi dòng vào out_path; chạy tiếp được nếu bị ngắt."""
    prompt = prompt or PROMPT
    out_path = Path(out_path)
    done = read_done(out_path)
    write_jsonl(out_path, done)                         # ghi lại sạch (bỏ dòng hỏng nếu có)
    done_ids = {r["image_id"] for r in done}
    todo = [e for e in entries if e["image_id"] not in done_ids]
    print(f"{out_path}: đã có {len(done_ids)} ảnh, cần chạy {len(todo)} ảnh")
    model.eval()
    t0 = time.time()
    with open(out_path, "a", encoding="utf-8") as f:
        for bi in range(0, len(todo), batch_size):
            batch = todo[bi:bi + batch_size]
            raws = generate_batch(model, processor, [load_image(e) for e in batch], prompt, max_new_tokens)
            for e, raw in zip(batch, raws):
                f.write(json.dumps({"image_id": e["image_id"], "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (bi // batch_size) % log_every == 0:
                print(f"  {bi + len(batch)}/{len(todo)} ảnh  {time.time() - t0:.0f}s")
    print(f"xong {len(todo)} ảnh trong {(time.time() - t0) / 60:.1f} phút")
    backup(out_path)
    return read_jsonl(out_path)


def score(pred_path, name, out_path, split="test", coord="norm1000", meta=None):
    """Gọi trọng tài scripts/score.py (không tự tính lại) và trả về JSON kết quả."""
    cmd = [sys.executable, "scripts/score.py", "--pred", str(pred_path), "--name", name,
           "--out", str(out_path), "--split", split, "--coord", coord]
    for k, v in (meta or {}).items():
        cmd += ["--meta", f"{k}={v}"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout + res.stderr)
    res.check_returncode()
    backup(out_path)
    return json.loads(Path(out_path).read_text(encoding="utf-8"))


# ==== GPU: E0 zero-shot trên 311 ảnh test
processor = load_processor()
model = load_base_model()
print(f"VRAM sau khi nạp model 4-bit: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
predict_split(model, processor, splits["test"], WORK / "e0.jsonl", PROMPT)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

In [ ]:
# Phân tích output thô của E0 (GUIDE bước 3 "Kiểm tra" + REPORT câu 2)
def gt_boxes(entry):
    """Box đúng theo thang 0–1000, cùng định dạng parse_output: list[{"label", "bbox"}]."""
    return [{"label": o["label"], "bbox": tuple(v * 1000 for v in yolo_to_pixels(o["yolo"], 1, 1))}
            for o in entry["objects"]]


def diagnose(raw, entry):
    """Gắn nhãn lỗi cho một output thô (một ảnh có thể có nhiều nhãn lỗi)."""
    tags, text = set(), (raw or "").strip()
    if "```" in text:
        tags.add("bọc ```json")
    parsed = parse_output(raw)
    if parsed is None:
        tags.add("JSON hỏng / sai schema")
        if text.count("[") > text.count("]") or text.count("{") > text.count("}"):
            tags.add("bị cắt cụt (hết max_new_tokens)")
        return tags
    body = re.sub(r"```(?:json)?", "", text).strip()
    if not (body.startswith("[") and body.endswith("]")):
        tags.add("chữ thừa ngoài JSON")
    if not parsed:
        tags.add("trả [] (bỏ sót tất cả)")
    if any(p["label"] not in CLASSES for p in parsed):
        tags.add("nhãn ngoài 3 lớp")
    if any(max(p["bbox"]) > 1000 for p in parsed):
        tags.add("toạ độ > 1000 (sai thang)")
    gts = gt_boxes(entry)
    for p in parsed:
        any_iou = max((iou_xyxy(p["bbox"], g["bbox"]) for g in gts), default=0.0)
        same_iou = max((iou_xyxy(p["bbox"], g["bbox"]) for g in gts if g["label"] == p["label"]), default=0.0)
        if any_iou < 0.1:
            tags.add("box bịa (IoU < 0.1 với mọi box đúng)")
        elif same_iou < 0.5 <= any_iou:
            tags.add("sai lớp (IoU ≥ 0.5 với box lớp khác)")
        elif same_iou < 0.5:
            tags.add("box lệch (0.1 ≤ IoU < 0.5)")
    return tags


def error_report(pred_path, split="test"):
    preds = {r["image_id"]: r["raw"] for r in read_jsonl(pred_path)}
    entries = splits[split]
    cnt = Counter(t for e in entries for t in diagnose(preds.get(e["image_id"]), e))
    print(f"Lỗi trong {pred_path} ({len(entries)} ảnh):")
    for t, n in cnt.most_common():
        print(f"  {n:4d} ảnh ({n / len(entries):5.1%})  {t}")
    return cnt


def coord_stats(pred_path):
    """Toạ độ lớn nhất model trả về: nếu x ≤ 640 và y ≤ 360 gần như luôn đúng thì nghi model trả pixel."""
    xs, ys = [], []
    for r in read_jsonl(pred_path):
        for p in parse_output(r["raw"]) or []:
            xs += [p["bbox"][0], p["bbox"][2]]
            ys += [p["bbox"][1], p["bbox"][3]]
    if xs:
        print(f"{len(xs) // 2} box | x: max {max(xs):.0f}, p95 {np.percentile(xs, 95):.0f} | "
              f"y: max {max(ys):.0f}, p95 {np.percentile(ys, 95):.0f} | y > 360: {np.mean(np.array(ys) > 360):.1%}")


backup(EVID / "e0.json")
for r in read_jsonl(WORK / "e0.jsonl")[:5]:
    print(r["image_id"], "→", repr(r["raw"][:400]), "\n")
E0_ERRORS = error_report(WORK / "e0.jsonl")
coord_stats(WORK / "e0.jsonl")
# Model gốc có đang trả pixel không? Chấm lại cùng output như thể là pixel (chỉ để kiểm tra, không nộp)
score(WORK / "e0.jsonl", "E0-as-pixel", WORK / "e0_as_pixel.json", coord="pixel")
del model
free_gpu()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

from functools import partial

from torch.utils.data import DataLoader, Dataset


def encode_example(processor, image, prompt, answer):
    user = user_message(image, prompt)
    assistant = {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    prompt_enc = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt")
    full_enc = processor.apply_chat_template([user, assistant], tokenize=True, add_generation_prompt=False,
                                             return_dict=True, return_tensors="pt")
    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), \
        "prompt không phải tiền tố của chuỗi đầy đủ"
    labels = full_enc["input_ids"].clone()
    labels[:, :n] = -100                        # không tính loss trên prompt + token ảnh: chỉ học câu trả lời
    enc = dict(full_enc)
    enc["labels"] = labels
    return enc


def collate(batch, pad_id):
    L = max(b["input_ids"].shape[1] for b in batch)

    def pad(t, value):                          # đệm phải tới độ dài L
        return torch.cat([t, t.new_full((t.shape[0], L - t.shape[1]), value)], dim=1)

    out = {k: torch.cat([pad(b[k], v) for b in batch])
           for k, v in (("input_ids", pad_id), ("attention_mask", 0), ("mm_token_type_ids", 0), ("labels", -100))}
    out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch])        # nối theo chiều 0, KHÔNG pad
    out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch])
    return out


class GroundingDataset(Dataset):
    """Mỗi phần tử: một ảnh train đã encode (input_ids, labels đã mask, pixel_values, ...)."""

    def __init__(self, entries, processor, coord="norm1000"):
        self.entries, self.processor, self.coord = entries, processor, coord
        self.prompt = make_prompt(coord)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i):
        e = self.entries[i]
        answer = objects_to_answer(e["objects"], self.coord, e["width"], e["height"])
        return encode_example(self.processor, load_image(e), self.prompt, answer)


# Kiểm tra (chỉ cần processor, không cần GPU)
processor = load_processor()
ds_check = GroundingDataset(splits["train"], processor)
ex = ds_check[0]
n_img = int((ex["input_ids"] == processor.image_token_id).sum())
print("số token ảnh:", n_img, "| image_grid_thw:", ex["image_grid_thw"].tolist(),
      "| tổng token:", ex["input_ids"].shape[1], "| token có loss:", int((ex["labels"] != -100).sum()))
assert n_img == 220, n_img
print("giải mã phần có loss:", repr(processor.tokenizer.decode(ex["labels"][ex["labels"] != -100])))
print("đáp án mong đợi     :", repr(objects_to_answer(splits["train"][0]["objects"])))
b = collate([ds_check[0], ds_check[1]], processor.tokenizer.pad_token_id)
print({k: tuple(v.shape) for k, v in b.items()})

# Độ dài đáp án (token): chọn max_new_tokens cho suy luận và max_completion_length cho E5
ans_len = {s: np.array([len(processor.tokenizer(objects_to_answer(r["objects"]))["input_ids"]) for r in rows])
           for s, rows in splits.items()}
for s, v in ans_len.items():
    print(f"{s:5s} token đáp án: trung vị {np.median(v):.0f}, p95 {np.percentile(v, 95):.0f}, max {v.max()}, "
          f"> 256: {(v > 256).sum()}, > 128: {(v > 128).sum()}")

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

import copy

import yaml
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoConfig

LLM_TARGETS = r"model\.language_model\.layers\.\d+\.(?:self_attn\.[qkvo]_proj|mlp\.(?:gate|up|down)_proj)"
VIT_TARGETS = r"model\.visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.linear_fc[12])"
PROJECTOR_KEYS = ("visual.merger", "visual.deepstack_merger_list")


def load_cfg(name):
    return yaml.safe_load(Path(f"configs/{name}.yaml").read_text(encoding="utf-8"))


def target_regex(targets):
    return LLM_TARGETS if targets == "llm" else f"(?:{LLM_TARGETS})|(?:{VIT_TARGETS})"


def apply_setup(model, cfg):
    for p in model.parameters():
        p.requires_grad_(False)
    model.enable_input_require_grads()            # cần cho gradient checkpointing khi đầu vào bị đóng băng
    regex = target_regex(cfg["lora"]["targets"])
    matched = [n for n, _ in model.named_modules() if re.fullmatch(regex, n)]
    lora = LoraConfig(r=cfg["lora"]["r"], lora_alpha=cfg["lora"]["alpha"], lora_dropout=cfg["lora"]["dropout"],
                      target_modules=regex, bias="none", task_type="CAUSAL_LM")
    pm = get_peft_model(model, lora)              # peft: target_modules dạng chuỗi = re.fullmatch trên tên module
    if cfg.get("train_projector"):
        for n, p in pm.named_parameters():
            if any(k in n for k in PROJECTOR_KEYS):
                assert p.is_floating_point(), f"{n} là trọng số 4-bit (dtype {p.dtype}): projector phải được giữ fp16 khi nạp"
                p.data = p.data.float()           # fp32: GradScaler không unscale được grad fp16, và update nhỏ bị mất ở fp16
                p.requires_grad_(True)
    n_lora = sum(p.numel() for n, p in pm.named_parameters() if p.requires_grad and "lora_" in n)
    n_proj = sum(p.numel() for n, p in pm.named_parameters() if p.requires_grad and any(k in n for k in PROJECTOR_KEYS))
    n_train = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    print(f"[{cfg['name']}] LoRA khớp {len(matched)} module | tham số huấn luyện {n_train:,} "
          f"(LoRA {n_lora:,} + projector {n_proj:,})")
    return pm, matched, n_train


def param_report(model_id=MODEL_ID):
    """Đếm tham số từng khối và số module LoRA khớp, trên model 'meta' (không tải trọng số, không cần GPU)."""
    with torch.device("meta"):
        m = Qwen3VLForConditionalGeneration(AutoConfig.from_pretrained(model_id))

    def count(pred):
        return sum(p.numel() for n, p in m.named_parameters() if pred(n))

    rep = {
        "total": count(lambda n: True),
        "vit_total (model.visual)": count(lambda n: n.startswith("model.visual.")),
        "vit_blocks+patch_embed (không gồm projector)": count(
            lambda n: n.startswith("model.visual.") and not any(k in n for k in PROJECTOR_KEYS)),
        "projector visual.merger": count(lambda n: "visual.merger" in n),
        "projector visual.deepstack_merger_list": count(lambda n: "visual.deepstack_merger_list" in n),
        "llm (model.language_model, gồm embedding)": count(lambda n: n.startswith("model.language_model.")),
        "embed_tokens (= lm_head, tied)": count(lambda n: "embed_tokens" in n),
    }
    names = [n for n, _ in m.named_modules()]
    rep["lora_modules llm"] = sum(bool(re.fullmatch(target_regex("llm"), n)) for n in names)
    rep["lora_modules llm+vit"] = sum(bool(re.fullmatch(target_regex("llm+vit"), n)) for n in names)
    for k, v in rep.items():
        print(f"  {k:48s} {v:>15,}")
    assert rep["lora_modules llm"] == 252 and rep["lora_modules llm+vit"] == 348
    del m
    return rep


PARAMS = param_report()

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

from safetensors.torch import load_file, save_file
from torch.optim.lr_scheduler import LambdaLR

RUNS = WORK / "runs"


def warmup_cosine(step, total, warmup):
    """Hệ số lr: tăng tuyến tính trong `warmup` bước đầu, sau đó cosine về 0."""
    if step < warmup:
        return (step + 1) / warmup
    return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


def projector_state(pm):
    return {n: p.detach().to(torch.float16).cpu().contiguous()
            for n, p in pm.named_parameters() if any(k in n for k in PROJECTOR_KEYS)}


def save_checkpoint(pm, out_dir, cfg):
    pm.save_pretrained(str(out_dir / "adapter"))                   # peft CHỈ lưu LoRA
    if cfg.get("train_projector"):                                  # projector huấn luyện đầy đủ → lưu riêng
        save_file(projector_state(pm), str(out_dir / "projector.safetensors"))
    backup(out_dir)


def infinite(loader):
    while True:
        yield from loader


def train(cfg, run_name, coord="norm1000", train_fraction=1.0, num_workers=None):
    """Huấn luyện một setup; lưu adapter (+ projector) mỗi 100 bước và run.json (meta + log loss)."""
    out_dir = RUNS / run_name
    if (out_dir / "run.json").exists():
        print(f"{run_name}: đã có run.json → bỏ qua huấn luyện")
        return json.loads((out_dir / "run.json").read_text(encoding="utf-8"))
    out_dir.mkdir(parents=True, exist_ok=True)
    random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    processor = load_processor()
    model = load_base_model()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    pm, matched, n_train = apply_setup(model, cfg)

    entries = splits["train"]
    if train_fraction < 1.0:
        entries = random.Random(cfg["seed"]).sample(entries, max(1, round(train_fraction * len(entries))))
    num_workers = (2 if IN_COLAB else 0) if num_workers is None else num_workers
    loader = DataLoader(GroundingDataset(entries, processor, coord), batch_size=cfg["batch_size"], shuffle=True,
                        drop_last=True, num_workers=num_workers, persistent_workers=num_workers > 0,
                        collate_fn=partial(collate, pad_id=processor.tokenizer.pad_token_id),
                        generator=torch.Generator().manual_seed(cfg["seed"]))
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    max_steps, accum = cfg["max_steps"], cfg["grad_accum"]
    sched = LambdaLR(opt, partial(warmup_cosine, total=max_steps, warmup=max(1, round(0.05 * max_steps))))
    cuda = DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    if cuda:
        torch.cuda.reset_peak_memory_stats()

    pm.train()
    batches, log, t0 = infinite(loader), [], time.time()
    for step in range(1, max_steps + 1):
        step_loss = 0.0
        for _ in range(accum):                                      # tích luỹ gradient
            batch = {k: v.to(DEVICE) for k, v in next(batches).items()}
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=cuda):
                loss = pm(**batch).loss / accum
            if not torch.isfinite(loss):
                raise RuntimeError(f"[{run_name}] loss không hữu hạn ở bước {step}: {loss.item()} (tràn số fp16?)")
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        sched.step()
        log.append({"step": step, "loss": step_loss, "lr": sched.get_last_lr()[0], "grad_norm": gnorm,
                    "minutes": (time.time() - t0) / 60})
        if step == 1 or step % 10 == 0:
            vram = f" | VRAM đỉnh {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB" if cuda else ""
            print(f"[{run_name}] bước {step}/{max_steps} loss={step_loss:.4f} lr={log[-1]['lr']:.2e} "
                  f"|g|={gnorm:.2f} {log[-1]['minutes']:.1f} phút{vram}")
        if step % 100 == 0 or step == max_steps:
            save_checkpoint(pm, out_dir, cfg)

    run = {"name": run_name, "config": cfg, "coord": coord, "train_fraction": train_fraction,
           "n_train_images": len(entries), "lora_modules": len(matched), "trainable_params": n_train,
           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if cuda else 0.0,
           "train_minutes": round((time.time() - t0) / 60, 2), "log": log}
    (out_dir / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=1), encoding="utf-8")
    backup(out_dir)
    print(f"[{run_name}] xong: {run['train_minutes']} phút, VRAM đỉnh {run['peak_vram_gb']} GB, "
          f"{n_train:,} tham số huấn luyện")
    del pm, model, opt, loader, batches
    free_gpu()
    return run


def plot_loss_curves(names=("e1", "e2", "e3"), fname="loss_curves.png"):
    fig, ax = plt.subplots(figsize=(9, 4))
    for n in names:
        p = RUNS / n / "run.json"
        if p.exists():
            log = json.loads(p.read_text(encoding="utf-8"))["log"]
            loss = np.array([r["loss"] for r in log])
            k = min(10, len(loss))
            smooth = np.convolve(loss, np.ones(k) / k, mode="valid")
            ax.plot([r["step"] for r in log], loss, alpha=0.25)
            ax.plot(np.arange(k, len(loss) + 1), smooth, label=f"{n.upper()} (TB trượt {k} bước)",
                    color=ax.lines[-1].get_color())
    ax.set_xlabel("bước tối ưu")
    ax.set_ylabel("loss (chỉ trên token trả lời)")
    ax.set_yscale("log")
    ax.set_title("Loss huấn luyện")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=110)
    plt.show()

In [ ]:
# ==== GPU: E1 — LoRA chỉ trên LLM (≈ 20–30 phút trên T4)
RUN_E1 = train(load_cfg("e1"), "e1")

In [ ]:
# ==== GPU: E2 — E1 + mở đầy đủ projector (merger + deepstack_merger_list)
RUN_E2 = train(load_cfg("e2"), "e2")

In [ ]:
# ==== GPU: E3 — E2 + LoRA trên attention/MLP của ViT
# T4 hết VRAM với batch 2 (gradient đi qua toàn bộ ViT + logits 151.936 từ vựng):
# dùng batch 1 × tích luỹ 8 → batch hiệu dụng vẫn là 8 như E1/E2 (GUIDE, "Khi gặp sự cố")
cfg_e3 = load_cfg("e3")
cfg_e3.update(batch_size=1, grad_accum=8)
RUN_E3 = train(cfg_e3, "e3")

In [ ]:
plot_loss_curves()

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

def load_for_inference(run_dir):
    """Model gốc 4-bit → nạp adapter LoRA → nạp lại projector đã huấn luyện (nếu setup có mở projector)."""
    run_dir = Path(run_dir)
    model = load_base_model()
    pm = PeftModel.from_pretrained(model, str(run_dir / "adapter"))
    proj = run_dir / "projector.safetensors"
    if proj.exists():
        state = load_file(str(proj))
        cur = dict(pm.named_parameters())
        k0 = next(iter(state))
        diff = (state[k0].float().cpu() - cur[k0].detach().float().cpu()).abs().mean().item()
        res = pm.load_state_dict(state, strict=False)
        assert not res.unexpected_keys, res.unexpected_keys[:5]
        print(f"nạp lại {len(state)} tensor projector ({sum(v.numel() for v in state.values()):,} tham số); "
              f"|Δ| trung bình so với projector gốc ở {k0}: {diff:.2e} (phải > 0)")
        assert diff > 0, "projector nạp vào giống hệt bản gốc → kiểm tra lại bước lưu"
    pm.eval()
    pm.config.use_cache = True
    return pm


def run_meta(run_name):
    run = json.loads((RUNS / run_name / "run.json").read_text(encoding="utf-8"))
    return {"trainable_params": run["trainable_params"], "peak_vram_gb": f"{run['peak_vram_gb']:.2f}",
            "train_minutes": f"{run['train_minutes']:.2f}"}, run


def quick_val(run_name, n=40, max_new_tokens=256):
    """Gỡ lỗi nhanh trên n ảnh val (không ghi bằng chứng)."""
    _, run = run_meta(run_name)
    entries = splits["val"][:n]
    processor = load_processor()
    pm = load_for_inference(RUNS / run_name)
    path = WORK / f"{run_name}_val{n}.jsonl"
    rows = predict_split(pm, processor, entries, path, make_prompt(run["coord"]), max_new_tokens=max_new_tokens)
    del pm
    free_gpu()
    r = evaluate(entries, {x["image_id"]: x["raw"] for x in rows}, coord=run["coord"], n_boot=200)
    print(f"[{run_name}] val[:{n}] valid={r['parse_valid_rate']:.3f} F1={r['micro']['f1']:.3f} "
          f"CI95={r['f1_ci95'][0]:.3f}-{r['f1_ci95'][1]:.3f}")
    return r


def evaluate_run(run_name, exp_name=None, max_new_tokens=256):
    """Suy luận trên test → work/<run>.jsonl rồi chấm bằng scripts/score.py kèm --meta từ run.json."""
    meta, run = run_meta(run_name)
    pred_path = WORK / f"{run_name}.jsonl"
    processor = load_processor()
    pm = load_for_inference(RUNS / run_name)
    predict_split(pm, processor, splits["test"], pred_path, make_prompt(run["coord"]), max_new_tokens=max_new_tokens)
    del pm
    free_gpu()
    return score(pred_path, exp_name or run_name.upper(), EVID / f"{run_name}.json", coord=run["coord"], meta=meta)


# ==== GPU: gỡ lỗi trên 40 ảnh val trước, rồi chấm test MỘT lần cho mỗi setup
quick_val("e1")
EV = {k: evaluate_run(k) for k in ("e1", "e2", "e3")}

In [ ]:
# Bảng tổng hợp + biểu đồ (đọc report/evidence/*.json — số liệu chỉ lấy từ trọng tài)
SETUP_DESC = {"e0": "zero-shot", "e1": "LoRA LLM", "e2": "+ projector", "e3": "+ LoRA ViT", "e5": "GRPO (reward IoU)"}


def load_evidence():
    return {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted(EVID.glob("e*.json"))}


def summary_table(ev):
    head = ("| Mã | Setup | Tham số huấn luyện | VRAM đỉnh (GB) | Phút | F1 | CI95 | FP/ảnh | "
            "F1 pothole | F1 crack | F1 manhole | parse_valid | P | R | meanIoU |")
    lines = [head, "|" + "---|" * 15]
    for k, d in ev.items():
        m, pc = d.get("meta", {}), d["per_class"]
        lines.append(
            f"| {d['name']} | {SETUP_DESC.get(k, k)} | {m.get('trainable_params', 0):,} | "
            f"{m.get('peak_vram_gb', '–')} | {m.get('train_minutes', '–')} | {d['micro']['f1']:.3f} | "
            f"{d['f1_ci95'][0]:.3f}–{d['f1_ci95'][1]:.3f} | {d['fp_per_image']:.2f} | "
            + " | ".join(f"{pc[c]['f1']:.3f}" for c in CLASSES)
            + f" | {d['parse_valid_rate']:.3f} | {d['micro']['precision']:.3f} | {d['micro']['recall']:.3f} | "
              f"{d['mean_iou']:.3f} |")
    md = "\n".join(lines)
    Path("report/summary_table.md").write_text(md + "\n", encoding="utf-8")
    backup("report/summary_table.md")
    return md


def plot_results(ev):
    keys = list(ev)
    names = [ev[k]["name"] for k in keys]
    f1 = np.array([ev[k]["micro"]["f1"] for k in keys])
    lo = np.array([ev[k]["f1_ci95"][0] for k in keys])
    hi = np.array([ev[k]["f1_ci95"][1] for k in keys])
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
    ax = axes[0]
    ax.errorbar(range(len(keys)), f1, yerr=[f1 - lo, hi - f1], fmt="o", capsize=5)
    for i, v in enumerate(f1):
        ax.annotate(f"{v:.3f}", (i, v), textcoords="offset points", xytext=(8, 0), fontsize=8)
    ax.set_xticks(range(len(keys)), names, rotation=20, fontsize=8)
    ax.set_ylabel("F1 @ IoU 0.5 (micro)")
    ax.set_title("F1 test và CI95 bootstrap")
    ax = axes[1]
    w = 0.8 / len(CLASSES)
    for j, c in enumerate(CLASSES):
        ax.bar(np.arange(len(keys)) + (j - 1) * w, [ev[k]["per_class"][c]["f1"] for k in keys], w,
               color=COLORS[c], label=c)
    ax.set_xticks(range(len(keys)), names, rotation=20, fontsize=8)
    ax.set_title("F1 từng lớp")
    ax.legend(fontsize=8)
    ax = axes[2]
    tr = [k for k in keys if ev[k].get("meta", {}).get("peak_vram_gb")]
    ax.bar(np.arange(len(tr)) - 0.2, [ev[k]["meta"]["peak_vram_gb"] for k in tr], 0.4, label="VRAM đỉnh (GB)")
    ax2 = ax.twinx()
    ax2.bar(np.arange(len(tr)) + 0.2, [ev[k]["meta"]["train_minutes"] for k in tr], 0.4, color="tab:orange",
            label="phút huấn luyện")
    ax.set_xticks(range(len(tr)), [ev[k]["name"] for k in tr], rotation=20, fontsize=8)
    ax.set_ylabel("GB")
    ax2.set_ylabel("phút")
    ax.set_title("Chi phí huấn luyện")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(FIG / "results_overview.png", dpi=110)
    plt.show()


EV = load_evidence()
print(summary_table(EV))
plot_results(EV)

In [ ]:
# Phân tích lỗi: ảnh minh hoạ đúng/sai, recall theo kích thước box, phân loại FP, và kiểm tra thang toạ độ
def match_image(gts, preds, thr=0.5):
    """Ghép tham lam như score.py. Trả (tp_pairs [(gi, pj, iou)], fp_idx, fn_idx)."""
    pairs = sorted(((iou_xyxy(g["bbox"], p["bbox"]), i, j) for i, g in enumerate(gts)
                    for j, p in enumerate(preds) if g["label"] == p["label"]), reverse=True)
    ug, up, tp = set(), set(), []
    for v, i, j in pairs:
        if v < thr:
            break
        if i not in ug and j not in up:
            ug.add(i)
            up.add(j)
            tp.append((i, j, v))
    return tp, [j for j in range(len(preds)) if j not in up], [i for i in range(len(gts)) if i not in ug]


def per_image(run_name, coord="norm1000"):
    preds = {r["image_id"]: r["raw"] for r in read_jsonl(WORK / f"{run_name}.jsonl")}
    out = []
    for e in splits["test"]:
        parsed = parse_output(preds.get(e["image_id"]))
        pr = [] if parsed is None else parsed
        if coord == "pixel":
            pr = [{"label": p["label"], "bbox": (p["bbox"][0] / IMG_W * 1000, p["bbox"][1] / IMG_H * 1000,
                                                  p["bbox"][2] / IMG_W * 1000, p["bbox"][3] / IMG_H * 1000)} for p in pr]
        g = gt_boxes(e)
        tp, fp, fn = match_image(g, pr)
        f1 = 2 * len(tp) / (2 * len(tp) + len(fp) + len(fn)) if g or pr else 1.0
        out.append({"entry": e, "gt": g, "pred": pr, "valid": parsed is not None, "tp": tp, "fp": fp, "fn": fn, "f1": f1})
    return out


def show_examples(run_name, n_good=4, n_bad=4, coord="norm1000", fname=None):
    rows = per_image(run_name, coord)
    good = sorted([r for r in rows if len(r["gt"]) >= 2], key=lambda r: (-r["f1"], -len(r["gt"])))[:n_good]
    bad = sorted(rows, key=lambda r: (r["f1"], -(len(r["fp"]) + len(r["fn"]))))[:n_bad]
    fig, axes = plt.subplots(2, max(n_good, n_bad), figsize=(5 * max(n_good, n_bad), 6.4))
    for ax, r in zip(list(axes[0]) + list(axes[1]), good + bad):
        e = r["entry"]
        ax.imshow(load_image(e))
        ax.axis("off")
        tp_g = {i for i, _, _ in r["tp"]}
        draw_boxes(ax, [(g["label"], box_to_pixels(g["bbox"])) for i, g in enumerate(r["gt"])],
                   color="lime", lw=3, show_label=False)
        draw_boxes(ax, [(r["pred"][j]["label"], box_to_pixels(r["pred"][j]["bbox"])) for _, j, _ in r["tp"]],
                   color="cyan", ls="--", lw=1.6)
        draw_boxes(ax, [(r["pred"][j]["label"], box_to_pixels(r["pred"][j]["bbox"])) for j in r["fp"]],
                   color="red", ls="--", lw=1.6)
        missed = Counter(r["gt"][i]["label"] for i in r["fn"])
        ax.set_title(f"{e['image_id']}  F1={r['f1']:.2f}\nTP={len(r['tp'])} FP={len(r['fp'])} FN={len(r['fn'])}"
                     + (f"  sót: {dict(missed)}" if missed else "") + ("" if r["valid"] else "  [JSON hỏng]"),
                     fontsize=8)
    fig.suptitle(f"{run_name.upper()} — hàng trên: dự đoán tốt, hàng dưới: dự đoán sai. "
                 "Xanh lá dày = box đúng (GT); xanh dương đứt = dự đoán khớp (TP); đỏ đứt = dự đoán sai (FP)",
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG / (fname or f"qualitative_{run_name}.png"), dpi=110)
    plt.show()


def compare_runs(run_names, image_ids, fname="compare_runs.png"):
    """Cùng một ảnh qua nhiều setup (vd E0 vs E1 vs tốt nhất)."""
    rows = {n: {r["entry"]["image_id"]: r for r in per_image(n)} for n in run_names}
    fig, axes = plt.subplots(len(image_ids), len(run_names), figsize=(5 * len(run_names), 3.1 * len(image_ids)))
    axes = np.atleast_2d(axes)
    for i, iid in enumerate(image_ids):
        for j, n in enumerate(run_names):
            r, ax = rows[n][iid], axes[i, j]
            ax.imshow(load_image(r["entry"]))
            ax.axis("off")
            draw_boxes(ax, [(g["label"], box_to_pixels(g["bbox"])) for g in r["gt"]], color="lime", lw=3, show_label=False)
            draw_boxes(ax, [(p["label"], box_to_pixels(p["bbox"])) for p in r["pred"]], ls="--", lw=1.5)
            ax.set_title(f"{n.upper()} | {iid} | TP={len(r['tp'])} FP={len(r['fp'])} FN={len(r['fn'])}", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=110)
    plt.show()


SIZE_BINS = [0, 0.3, 1, 3, 100]          # % diện tích khung hình


def recall_by_size(run_name, coord="norm1000", fname=None):
    rows = per_image(run_name, coord)
    hit = defaultdict(lambda: [0, 0])
    for r in rows:
        matched = {i for i, _, _ in r["tp"]}
        for i, g in enumerate(r["gt"]):
            x1, y1, x2, y2 = g["bbox"]
            area = (x2 - x1) * (y2 - y1) / 1e4                         # % khung hình
            b = np.digitize(area, SIZE_BINS[1:-1])
            hit[(g["label"], b)][0] += i in matched
            hit[(g["label"], b)][1] += 1
    labels = [f"{SIZE_BINS[b]}–{SIZE_BINS[b + 1]}%" for b in range(len(SIZE_BINS) - 1)]
    fig, ax = plt.subplots(figsize=(9, 4))
    w = 0.27
    for j, c in enumerate(CLASSES):
        vals = [hit[(c, b)][0] / hit[(c, b)][1] if hit[(c, b)][1] else 0 for b in range(len(labels))]
        bars = ax.bar(np.arange(len(labels)) + (j - 1) * w, vals, w, color=COLORS[c], label=c)
        ax.bar_label(bars, labels=[f"n={hit[(c, b)][1]}" for b in range(len(labels))], fontsize=6)
    ax.set_xticks(range(len(labels)), labels)
    ax.set_xlabel("diện tích box đúng (% khung hình)")
    ax.set_ylabel("recall @ IoU 0.5")
    ax.set_title(f"{run_name.upper()}: recall theo kích thước box và lớp")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / (fname or f"recall_by_size_{run_name}.png"), dpi=110)
    plt.show()
    return dict(hit)


def fp_breakdown(run_name, coord="norm1000"):
    """FP thuộc loại nào: sai lớp / lệch vị trí / bịa trên nền."""
    cnt = Counter()
    for r in per_image(run_name, coord):
        for j in r["fp"]:
            p = r["pred"][j]
            any_iou = max((iou_xyxy(p["bbox"], g["bbox"]) for g in r["gt"]), default=0.0)
            same_iou = max((iou_xyxy(p["bbox"], g["bbox"]) for g in r["gt"] if g["label"] == p["label"]), default=0.0)
            kind = ("sai lớp" if any_iou >= 0.5 > same_iou else "lệch vị trí (0.1≤IoU<0.5)" if any_iou >= 0.1
                    else "bịa trên nền (IoU<0.1)")
            cnt[(p["label"], kind)] += 1
    for (lab, kind), n in sorted(cnt.items()):
        print(f"  FP {lab:8s} {kind:28s} {n}")
    return cnt


EV = load_evidence()
BEST = max([k for k in ("e1", "e2", "e3") if k in EV], key=lambda k: EV[k]["micro"]["f1"])
print("setup có F1 cao nhất trong E1–E3:", BEST)
show_examples(BEST)                                        # ≥ 6 ảnh đúng/sai có box dự đoán và box đúng
show_examples("e0", fname="qualitative_e0.png")
ids = [r["entry"]["image_id"] for r in sorted(per_image(BEST), key=lambda r: -len(r["gt"]))[:3]]
compare_runs(["e0", "e1", BEST] if BEST != "e1" else ["e0", "e1"], ids)
RECALL_SIZE = recall_by_size(BEST)
FP_KIND = fp_breakdown(BEST)
error_report(WORK / f"{BEST}.jsonl")
# Câu 7–8: hiệu F1 giữa các setup với CI95 bootstrap GHÉP CẶP (cùng ảnh được lấy mẫu lại cho cả hai setup)
from lab.stats import paired_bootstrap, per_image_counts


def compare(a, b):
    rows = [per_image_counts(splits["test"], {r["image_id"]: r["raw"] for r in read_jsonl(WORK / f"{k}.jsonl")},
                             EV[k].get("coord", "norm1000")) for k in (a, b)]
    r = paired_bootstrap(*rows)
    print(f"F1({a}) − F1({b}) = {r['diff']:+.3f}  CI95 ghép cặp [{r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f}]  "
          f"P(A>B) = {r['p_a_better']:.2f}")
    return r


PAIRED = {(a, b): compare(a, b) for a, b in (("e1", "e0"), ("e2", "e1"), ("e3", "e2"), ("e3", "e1"))}
# Câu 3: chấm output của setup tốt nhất với thang SAI (pixel) — chỉ để kiểm chứng, không phải bằng chứng nộp
score(WORK / f"{BEST}.jsonl", f"{BEST.upper()}-as-pixel", WORK / f"{BEST}_as_pixel.json", coord="pixel")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9

# ---------- E4: ablation, đổi ĐÚNG MỘT biến so với setup tốt nhất của E1–E3
E4_VARIABLE = "coord_scale"          # chọn một: "rank" | "data_size" | "coord_scale"
E4_VALUES = {"rank": [4, 64], "data_size": [0.1, 1.0], "coord_scale": ["norm1000", "pixel"]}
E4_BASE = BEST                        # setup gốc để ablation (in ở cell trên)


def e4_run(variable, value, base=E4_BASE):
    # lấy ĐÚNG cấu hình đã chạy của setup gốc từ run.json (vd E3 chạy batch 1 × tích luỹ 8 vì hết VRAM),
    # không đọc lại yaml — nếu không E4 sẽ đổi thêm batch size ngoài biến đang khảo sát
    cfg = copy.deepcopy(run_meta(base)[1]["config"])
    coord, frac = "norm1000", 1.0
    if variable == "rank":
        cfg["lora"]["r"], cfg["lora"]["alpha"] = value, 2 * value
    elif variable == "data_size":
        frac = value
    elif variable == "coord_scale":
        coord = value
    else:
        raise ValueError(variable)
    tag, exp_name = f"e4_{variable}_{value}", f"E4 {variable}={value} ({base.upper()})"
    cfg["name"] = exp_name
    if (variable, value) in (("data_size", 1.0), ("coord_scale", "norm1000")):
        # giá trị này CHÍNH LÀ setup gốc → chấm lại đúng dự đoán của setup gốc dưới tên E4, không huấn luyện lại
        meta, _ = run_meta(base)
        shutil.copy2(WORK / f"{base}.jsonl", WORK / f"{tag}.jsonl")
        return score(WORK / f"{tag}.jsonl", exp_name, EVID / f"{tag}.json", coord=coord, meta=meta)
    train(cfg, tag, coord=coord, train_fraction=frac)
    return evaluate_run(tag, exp_name)


def plot_e4(variable, values):
    ev = {v: json.loads((EVID / f"e4_{variable}_{v}.json").read_text(encoding="utf-8")) for v in values}
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(CLASSES) + 1)
    w = 0.38
    for i, (v, d) in enumerate(ev.items()):
        vals = [d["micro"]["f1"]] + [d["per_class"][c]["f1"] for c in CLASSES]
        bars = ax.bar(x + (i - 0.5) * w, vals, w, label=f"{variable}={v}")
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
        ax.errorbar(x[0] + (i - 0.5) * w, d["micro"]["f1"],
                    yerr=[[d["micro"]["f1"] - d["f1_ci95"][0]], [d["f1_ci95"][1] - d["micro"]["f1"]]],
                    fmt="none", ecolor="black", capsize=5)
    ax.set_xticks(x, ["F1 micro (± CI95)"] + [f"F1 {c}" for c in CLASSES])
    ax.set_title(f"E4: ablation {variable} trên {E4_BASE.upper()}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / f"e4_{variable}.png", dpi=110)
    plt.show()


# ==== GPU: E4 (giá trị trùng setup gốc chỉ chấm lại, giá trị còn lại huấn luyện ≈ 25 phút)
E4 = {v: e4_run(E4_VARIABLE, v) for v in E4_VALUES[E4_VARIABLE]}
plot_e4(E4_VARIABLE, E4_VALUES[E4_VARIABLE])
EV = load_evidence()
print(summary_table(EV))

### 8b. E5 (tuỳ chọn, +10): GRPO với reward IoU
Reward nằm trong `lab/reward.py`, test trong `lab/test_reward.py` (viết và chạy **trước** GRPO).

In [ ]:
# ---------- E5: test reward trước, sau đó GRPO khởi tạo từ adapter tốt nhất
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "lab"], capture_output=True, text=True).stdout[-400:])
from lab.reward import grpo_reward_func, iou_reward

RUN_E5 = False                        # phần tuỳ chọn (+10): đặt True để chạy GRPO


def build_grpo_dataset(entries, processor, max_answer_tokens=100, n=None, seed=42):
    """Dataset hội thoại cho trl: prompt (chuỗi) + cột image + cột gt (JSON đáp án 0–1000) cho hàm reward.
    Bỏ ảnh có đáp án đúng dài hơn max_answer_tokens: với max_completion_length=128, đáp án đó bị cắt
    → JSON hỏng → reward 0 dù model trả đúng, và tạo áp lực "trả ít box hơn" (một dạng reward hacking)."""
    from datasets import Dataset, Image as HFImage
    rows = []
    for e in entries:
        ans = objects_to_answer(e["objects"])
        if len(processor.tokenizer(ans)["input_ids"]) <= max_answer_tokens:
            rows.append({"prompt": [{"role": "user", "content": PROMPT}], "image": str(DATA / e["image_path"]),
                         "gt": ans})
    random.Random(seed).shuffle(rows)
    rows = rows[:n] if n else rows
    print(f"GRPO dataset: {len(rows)}/{len(entries)} ảnh (đáp án ≤ {max_answer_tokens} token)")
    return Dataset.from_list(rows).cast_column("image", HFImage())


def run_grpo(base, out_name="e5"):
    from trl import GRPOConfig, GRPOTrainer
    out_dir = RUNS / out_name
    if (out_dir / "run.json").exists():
        print(f"{out_name}: đã có run.json → bỏ qua")
        return json.loads((out_dir / "run.json").read_text(encoding="utf-8"))
    g = yaml.safe_load(Path("configs/e5.yaml").read_text(encoding="utf-8"))["grpo"]
    processor = load_processor()
    model = load_base_model()
    pm = PeftModel.from_pretrained(model, str(RUNS / base / "adapter"), is_trainable=True)
    proj = RUNS / base / "projector.safetensors"
    if proj.exists():                                   # projector đã SFT của setup gốc (đóng băng trong GRPO)
        assert not pm.load_state_dict(load_file(str(proj)), strict=False).unexpected_keys
    for n_, p in pm.named_parameters():
        p.requires_grad_("lora_" in n_)
    n_train = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    cuda = DEVICE == "cuda"
    args = GRPOConfig(
        output_dir=str(out_dir / "trainer"), learning_rate=g["learning_rate"], beta=g["beta"],
        num_generations=g["num_generations"], max_completion_length=g["max_completion_length"],
        max_steps=g["max_steps"], per_device_train_batch_size=g["num_generations"], gradient_accumulation_steps=2,
        use_vllm=False, fp16=cuda, bf16=False, use_cpu=not cuda, gradient_checkpointing=cuda,   # T4 không có bf16
        gradient_checkpointing_kwargs={"use_reentrant": False}, logging_steps=1, save_strategy="no",
        report_to="none", seed=42, remove_unused_columns=False, dataloader_num_workers=0)
    trainer = GRPOTrainer(model=pm, reward_funcs=[grpo_reward_func], args=args, processing_class=processor,
                          train_dataset=build_grpo_dataset(splits["train"], processor))
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    out_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(out_dir / "adapter"))
    if proj.exists():
        shutil.copy2(proj, out_dir / "projector.safetensors")
    run = {"name": out_name, "base": base, "config": g, "coord": "norm1000", "trainable_params": n_train,
           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if cuda else 0.0,
           "train_minutes": round((time.time() - t0) / 60, 2), "log": trainer.state.log_history}
    (out_dir / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    backup(out_dir)
    del trainer, pm, model
    free_gpu()
    return run


def plot_grpo(run, fname="grpo_curves.png"):
    log = [r for r in run["log"] if "reward" in r]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    axes[0].plot([r["step"] for r in log], [r["reward"] for r in log], label="reward trung bình")
    if "reward_std" in log[0]:
        axes[0].plot([r["step"] for r in log], [r["reward_std"] for r in log], label="std trong nhóm")
    axes[0].set_xlabel("bước GRPO")
    axes[0].legend()
    axes[0].set_title("Reward IoU trong GRPO")
    key = next((k for k in ("completions/mean_length", "completion_length") if k in log[0]), None)
    if key:
        axes[1].plot([r["step"] for r in log], [r[key] for r in log])
        axes[1].set_title("Độ dài completion (token) — co về 0 là dấu hiệu trả []")
        axes[1].set_xlabel("bước GRPO")
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=110)
    plt.show()


def hacking_stats(run_name):
    """Dấu hiệu reward hacking: tỉ lệ trả [], số box/ảnh, diện tích box trung bình, FP/ảnh."""
    rows = per_image(run_name)
    n_box = [len(r["pred"]) for r in rows]
    areas = [(p["bbox"][2] - p["bbox"][0]) * (p["bbox"][3] - p["bbox"][1]) / 1e4 for r in rows for p in r["pred"]]
    s = {"run": run_name, "frac_empty": float(np.mean([r["valid"] and not r["pred"] for r in rows])),
         "boxes_per_img": float(np.mean(n_box)), "gt_per_img": float(np.mean([len(r["gt"]) for r in rows])),
         "mean_box_area_pct": float(np.mean(areas)) if areas else 0.0,
         "max_box_area_pct": float(np.max(areas)) if areas else 0.0,
         "mean_reward": float(np.mean([iou_reward(json.dumps([{"label": p["label"], "bbox_2d": list(p["bbox"])}
                                                               for p in r["pred"]]) if r["valid"] else None, r["gt"])
                                       for r in rows]))}
    print(s)
    return s


# ==== GPU: E5 GRPO (≈ 60 bước) rồi suy luận + chấm test
if RUN_E5:
    RUN_GRPO = run_grpo(BEST)
    plot_grpo(RUN_GRPO)
    EV["e5"] = evaluate_run("e5", "E5")
    HACK = [hacking_stats(BEST), hacking_stats("e5")]
    show_examples("e5", fname="qualitative_e5.png")
    EV = load_evidence()
    print(summary_table(EV))

In [ ]:
# Kiểm tra bài nộp + gói kết quả lên Drive để tải về máy (report/ + run.json, KHÔNG gồm trọng số)
print(subprocess.run([sys.executable, "scripts/check_submission.py"], capture_output=True, text=True).stdout)
if DRIVE_DIR is not None:
    pack = Path("/content/day10_results")
    if pack.exists():
        shutil.rmtree(pack)
    shutil.copytree("report", pack / "report")
    (pack / "work").mkdir(parents=True)
    for p in WORK.glob("*.jsonl"):                     # dự đoán thô: để chạy lab/fill_report.py (token paired) ở máy
        shutil.copy2(p, pack / "work" / p.name)
    for p in RUNS.glob("*/run.json"):
        (pack / "runs" / p.parent.name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, pack / "runs" / p.parent.name / "run.json")
    print(shutil.make_archive(str(DRIVE_DIR / "day10_results"), "zip", pack))